# 4. Generación de Indicadores

**Objetivo:** construir el *universo* (RNPDNO + BANCI cruzados y con banderas derivadas)
y dejarlo cacheado para que los notebooks de PPT lo consuman.

Este notebook ya **no es el pipeline**: la lógica vive en `src/pipeline/` y aquí sólo se
la invoca. Eso permite que `6_PPT_Reporte_Diario` y cualquier otro notebook de PPT
ejecuten la misma lógica con **configuraciones distintas**, sin copiar celdas y sin
volver a correr el matching.

## Las tres capas

| Capa | Qué hace | Costo | Llave de caché |
|---|---|---|---|
| **A. Matching** | carga fuentes, 3 motores, cascada, reglas | minutos | `cfg.clave_matching` |
| **B. Derivadas** | banderas `is_*`, `fecha_dt`, `cve_ent`, curación de columnas | segundos | `cfg.clave_universo` |
| **C. Rebanadas** | máscaras por base / período / estatus / subfiltro / **entidad** | µs | — |

Cambiar un período de rebanado, un estatus o una entidad **no invalida nada**: son máscaras
booleanas. Todo lo que puede mover un cruce sí da otra llave, y el caché viejo deja de
usarse: un campo del perfil, `config.py` (también los valores que dependen del entorno), el
código de los motores o cualquier insumo que se haya leído. Detalle en `src/pipeline/huella.py`.

> **La entidad dejó de filtrarse antes del matching.** El universo se construye siempre
> nacional y `cve_ent` viaja como columna, así que una sola corrida de motores alimenta
> las 32 PPT estatales.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

from src.settings import Settings, perfiles_disponibles
from src.pipeline import Universo
from src.slices import Slice, expandir_entidades

print("Perfiles disponibles:", perfiles_disponibles())

## 1. Configuración

La configuración es un objeto inmutable que se lee de `perfiles/<nombre>.toml`.
Ya no se edita `config.py` ni se hace `importlib.reload`: dos configuraciones pueden
coexistir en este mismo kernel.

```python
cfg_a = Settings.desde_perfil("reporte_diario")
cfg_b = Settings.desde_perfil("estatal").con(estado=17)   # ambas vivas a la vez
```

In [ ]:
cfg = Settings.desde_perfil("reporte_diario")
print(cfg.resumen())

## 2. Construcción del universo

La primera corrida ejecuta los 3 motores (minutos) y guarda el resultado. Las siguientes
leen del caché en segundos, mientras la llave coincida y ningún insumo haya cambiado; si no,
se recalcula solo.

- `forzar_matching=True` → reejecuta los motores.
- `forzar=True` → reejecuta todo.
- `limpiar_cache(cfg)` → borra el caché en disco.

> Si se edita código de `src/` con este kernel abierto, el resultado se calcula pero **no se
> guarda** en caché: el kernel podría estar corriendo la versión anterior. Hay que reiniciarlo.

In [ ]:
u = Universo.construir(cfg)

print()
print(u.resumen().to_string(index=False))

In [ ]:
# Aportación de cada motor a la cascada
print(u.auditoria.to_string(index=False))

## 3. Rebanadas

Cada notebook de PPT declara **sólo** las rebanadas que necesita. Ya no se generan
todas las combinaciones posibles de nombres.

`Slice.nombre` respeta la nomenclatura de `catalogo_abreviaturas.md`, con el token
nuevo `ent<NN>` para la entidad (`reg_rn_01oct24-max_des_ent17`).

In [ ]:
REQUERIDOS = [
    Slice("rn",       "01oct24-max"),
    Slice("rn",       "01oct24-max", estatus="des"),
    Slice("rn-si-ba", "01oct24-max", estatus="des"),
]

# Conteos: no materializan ningún DataFrame.
for s in REQUERIDOS:
    print(f"  {s.nombre:<40} {u.contar(s):>10,}")

In [ ]:
# El DataFrame sólo cuando se necesita. materializar=True lo escribe además a Parquet
# (útil para entregarlo a R o Excel; para consumo en Python no hace falta).
df = u.slice(Slice("rn-si-ba", "01oct24-max", estatus="des"))
print(df.shape)
df.head()

In [ ]:
# Registros por tipo de match en reg_rn-si-ba_01oct24-max_des.
# `tipo_match` lista TODOS los métodos que alcanzaron ese FUB, en el orden de la
# cascada ("estandar, manual" = lo resolvieron los dos). Nunca es 'sin_match' en
# esta rebanada porque la base 'rn-si-ba' ya exige in_banci.
s_match = Slice("rn-si-ba", "01oct24-max", estatus="des")
df_match = u.slice(s_match)

conteo_match = (
    df_match["tipo_match_derivada_rnpdno_banci"]
    .value_counts(dropna=False)
    .rename_axis("tipo_match")
    .reset_index(name="registros")
)
conteo_match = conteo_match[conteo_match["registros"] > 0]
conteo_match["pct"] = (100 * conteo_match["registros"] / len(df_match)).round(1)

n_fubs = df_match["fub_rnpdno"].nunique()

print(f"{s_match.nombre}: {len(df_match):,} registros | {n_fubs:,} FUBs únicos")
print(conteo_match.to_string(index=False))

# Y el total por método, contando un FUB en cada método que lo sostiene
# (las combinaciones suman más que el total: un FUB puede aparecer en dos).
print("\nCobertura por método (un FUB cuenta en cada método que lo sostiene):")
for metodo in ("estandar", "manual", "refinado"):
    n = df_match["tipo_match_derivada_rnpdno_banci"].str.contains(metodo, regex=False).sum()
    print(f"  {metodo:<12} {n:>6,}  ({100*n/len(df_match):.1f}%)")





### 3.1 Exportar rebanadas a Parquet, enriquecidas

El universo ya trae la entidad (`cve_ent`, `entidad`), pero del municipio sólo
conserva `idmunicipio`: la clave cruda del RNPDNO, que llega como `float` (`75.0`), trae
huecos y **sólo es única dentro de su estado**. Para entregarla afuera se le pegan tres
columnas al final, siguiendo la convención de `exportacion_llaves.py`:

| Columna | Qué es |
|---|---|
| `cve_mun` | clave del municipio, entera y sin el `.0`; única **dentro** de la entidad |
| `cvegeo` | clave INEGI de 5 dígitos (`cve_ent` + `cve_mun`), única a nivel nacional |
| `municipio` | nombre del municipio, resuelto con el catálogo del INEGI |

`fechacaptura` y `fechalocaliza` ya **no** se pegan aquí: viajan en el universo desde que
se agregaron al mapeo del ETL (`RNPDNO_COLUMN_MAPPING`) y a `COLS_RN` de la CAPA B, porque
casi todos los entregables las piden. Salen con el nombre crudo del RNPDNO, igual que
`fechahechos_rnpdno` o `idmunicipio`.
Lo mismo `expedientes_rnpdno`: el expediente tal como viene, que va en `COLS_RN` para
que lo lleve toda base exportada con datos del RNPDNO, también las del notebook 6.

> **`fechalocaliza` sale vacía en las rebanadas `_des`, y es correcto.** En el origen ningún
> registro `DESAPARECIDA` o `NO LOCALIZADA` la trae —son justo las personas que no han sido
> localizadas—, así que en `reg_rn_min-max_des` y `reg_rn_01oct24-max_des` la columna es nula
> al 100%. Donde informa es en `_loc`: ahí la trae el 68.5%.

Se derivan **aquí** y no en la CAPA B a propósito: son tres columnas que sólo consumen
los entregables, no las rebanadas ni las PPT.

Los registros sin `idmunicipio` salen con las tres columnas **nulas, nunca en cero**; en el
corte actual van del 0.5% (`reg_rn_01oct24-max_des`) al 5.2% (`reg_rn_min-max_des`). Un
puñado más trae una clave que el catálogo no reconoce (municipios creados después del corte
del catálogo): ésos conservan su código y se quedan sin nombre.

In [ ]:
import config

# Catálogo INEGI: CVEGEO (5 dígitos) → nombre del municipio.
_cat_mun = pd.read_csv(config.CATALOGO_MUNICIPIOS_FILE, dtype=str, encoding="utf-8")
MAPA_MUNICIPIOS = dict(zip(_cat_mun["CVEGEO"], _cat_mun["NOM_MUN"]))


def con_claves_geo(df: pd.DataFrame) -> pd.DataFrame:
    """Añade clave de municipio, CVEGEO y nombre del municipio al final del DataFrame.

    Llevan el sufijo de la base de la que salen (`_derivada_rnpdno` o `_derivada_banci`).

    Sirve para las bases `rn*` (columna `idmunicipio_derivada_rnpdno`) y `ba*`
    (`id_mun_hchos_banci`).
    """
    col_mun = next((c for c in ("idmunicipio_derivada_rnpdno", "id_mun_hchos_banci")
                    if c in df.columns), None)
    if col_mun is None:
        raise KeyError(f"Sin columna de municipio. Columnas: {sorted(df.columns)}")

    sfx = "derivada_rnpdno" if col_mun.endswith("_rnpdno") else "derivada_banci"
    col_ent = "idestado_derivada_rnpdno" if sfx == "derivada_rnpdno" else "cve_entidad_derivada_banci"
    out = df.copy()
    ent = pd.to_numeric(out[col_ent], errors="coerce").astype("Int64")
    mun = pd.to_numeric(out[col_mun], errors="coerce").astype("Int64")

    # Sin entidad O sin municipio no hay CVEGEO: se deja nulo en vez de inventar "00000".
    completo = ent.notna() & mun.notna()
    cvegeo = pd.Series(pd.NA, index=out.index, dtype="string")
    cvegeo[completo] = (ent[completo].astype(int).map("{:02d}".format)
                        + mun[completo].astype(int).map("{:03d}".format))

    out[f"cve_mun_{sfx}"] = mun
    out[f"cvegeo_{sfx}"] = cvegeo
    out[f"nombre_municipio_{sfx}"] = cvegeo.map(MAPA_MUNICIPIOS).astype("string")
    return out


A_EXPORTAR = [
    Slice("rn", "min-max"),   # reg_rn_min-max
    Slice("rn", "min-max",     estatus="des"),   # reg_rn_min-max_des
    Slice("rn", "min-max",     estatus="loc"),   # reg_rn_min-max_loc
    Slice("rn", "01oct24-max", estatus="des"),   # reg_rn_01oct24-max_des
    # `sfval` completo ("Sin carpeta de investigación") y su rama con folio capturado:
    #     sfval_sfvac + sfval_cfvac = sfval
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval"),         # reg_rn-no-ba_01oct24-max_des_sfval
    Slice("rn-no-ba", "01oct24-max", estatus="des", subfiltro="sfval_sfvac"),   # reg_rn-no-ba_01oct24-max_des_sfval_sfvac
    Slice("rn-si-ba", "01oct24-max", estatus="des"),   # reg_rn-si-ba_01oct24-max_des
]

# `materializar=True` no se usa aquí: escribiría la rebanada ANTES de enriquecerla.
# Se rebana, se enriquece y se escribe con el mismo nombre canónico.
cfg.resultados_dir.mkdir(parents=True, exist_ok=True)
exportados = {}
for s in A_EXPORTAR:
    df_s = con_claves_geo(u.slice(s))
    destino = cfg.resultados_dir / f"{s.nombre}.parquet"
    df_s.to_parquet(destino, index=False, engine="pyarrow")
    exportados[s.nombre] = df_s
    print(f"💾 {s.nombre:<24} {len(df_s):>8,} registros | "
          f"{df_s['cvegeo_derivada_rnpdno'].isna().sum():>6,} sin municipio | "
          f"{df_s['fechalocaliza_derivada_rnpdno'].notna().sum():>7,} con fecha de localización "
          f"→ {destino.name}")

exportados["reg_rn_min-max_loc"][
    ["fub_rnpdno", "idestado_derivada_rnpdno", "nombre_entidad_derivada_rnpdno",
     "cve_mun_derivada_rnpdno", "cvegeo_derivada_rnpdno", "nombre_municipio_derivada_rnpdno",
     "fechacaptura_derivada_rnpdno", "fechalocaliza_derivada_rnpdno"]
].head()

## 4. Exportación de indicadores

Genera el CSV que consumen las plantillas PPTX.

In [ ]:
u.exportar_indicadores(REQUERIDOS)

## 5. Verificación y exploración

Todo lo de abajo es opcional: sirve para inspeccionar el universo sin recalcular nada.

In [ ]:
# Desglose por entidad de una misma rebanada: 33 máscaras booleanas, un solo universo.
base = Slice("rn", "01oct24-max", estatus="des")
por_ent = pd.DataFrame(
    [{"entidad": s.entidad_texto, "registros": u.contar(s)} for s in expandir_entidades(base)]
).sort_values("registros", ascending=False)
por_ent.head(10)

In [ ]:
# Composición de los matches por motor
df_m = u.ba[u.ba["in_rnpdno_derivada_rnpdno_banci"]]
print(df_m["tipo_match_derivada_rnpdno_banci"].value_counts())
print()
print("FUBs únicos matcheados:", u.meta["n_fubs_matcheados"])